# Prepare Documentation
## Final material for the Technical Analysis Document

1. Print the **outline** of the technical document (7 sections, ~10 pages)
2. Generate **`RESULTS_SUMMARY.md`** with real metrics, key findings, recommendations
3. Export **`export_for_report.json`** with all unified metrics

In [4]:
# ── Cell 1: Outline of the Technical Analysis Document ──────────────
outline = """
═══════════════════════════════════════════════════════════════════
 TECHNICAL ANALYSIS DOCUMENT — OUTLINE (~10 pages)
 Plant Disease Detection: Comparative Analysis of 4 Approaches
═══════════════════════════════════════════════════════════════════

1. PROBLEM STATEMENT                                       (~1.5 pages)
   1.1 Motivation (importance of plant disease detection)
   1.2 Dataset: New Plant Diseases Dataset (augmented PlantVillage)
       — 87,867 images, 38 classes
   1.3 Goal: compare 4 paradigms on the same task

2. METHODOLOGY                                             (~3 pages)
   2.1 V1 — HOG features + SVM (RBF kernel)
   2.2 V2 — Custom CNN from scratch (4 Conv+BN+ReLU+Pool blocks)
   2.3 V3 — ResNet50 Transfer Learning (freeze 1-3, fine-tune layer4)
   2.4 V4 — DINOv3 SSL Foundation Model + Linear Probe / k-NN
   2.5 Preprocessing: split ≈80/10/10 (valid/ split 50/50, seed=42),
       ImageNet normalization

3. EXPERIMENTAL SETUP                                      (~1 page)
   3.1 Hardware: Apple Silicon MPS / consumer GPU
   3.2 Hyperparameters per version (batch, lr, epochs, patience)
   3.3 Metrics: Accuracy, Precision, Recall, F1 (weighted)
   3.4 Reproducible code: notebooks 00 → 06, fixed seed

4. RESULTS                                                 (~2.5 pages)
   4.1 Test set metrics table (V1 vs V2 vs V3 vs V4)
   4.2 Confusion matrices for each version
   4.3 Training curves V2 vs V3 (V1 single-shot, V4 zero-train)
   4.4 Time/parameters/accuracy trade-off
   4.5 Discussion: why TL beats the custom CNN; why frozen SSL is
       surprisingly competitive with zero fine-tuning

5. FAILURE ANALYSIS                                        (~1 page)
   5.1 Hardest classes (per-class F1)
   5.2 Common confusion patterns (visually similar leaves)
   5.3 Paradigm-specific failure modes

6. ETHICAL & PRIVACY CONSIDERATIONS                        (~0.5 page)
   6.1 Dataset bias (controlled backgrounds, limited geography)
   6.2 Deployment in agriculture: who accesses the farm data?
   6.3 Computational cost and environmental impact

7. CONCLUSIONS                                             (~0.5 page)
   7.1 Best practice: TL (V3) for production, V4 for new domains
   7.2 Limitations and future directions
═══════════════════════════════════════════════════════════════════
"""
print(outline)


═══════════════════════════════════════════════════════════════════
 TECHNICAL ANALYSIS DOCUMENT — OUTLINE (~10 pages)
 Plant Disease Detection: Comparative Analysis of 4 Approaches
═══════════════════════════════════════════════════════════════════

1. PROBLEM STATEMENT                                       (~1.5 pages)
   1.1 Motivation (importance of plant disease detection)
   1.2 Dataset: New Plant Diseases Dataset (augmented PlantVillage)
       — 87,867 images, 38 classes
   1.3 Goal: compare 4 paradigms on the same task

2. METHODOLOGY                                             (~3 pages)
   2.1 V1 — HOG features + SVM (RBF kernel)
   2.2 V2 — Custom CNN from scratch (4 Conv+BN+ReLU+Pool blocks)
   2.3 V3 — ResNet50 Transfer Learning (freeze 1-3, fine-tune layer4)
   2.4 V4 — DINOv3 SSL Foundation Model + Linear Probe / k-NN
   2.5 Preprocessing: split ≈80/10/10 (valid/ split 50/50, seed=42),
       ImageNet normalization

3. EXPERIMENTAL SETUP                                

In [5]:
# ── Cell 2: Generate RESULTS_SUMMARY.md with real metrics ──────────────
import json
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path("..").resolve()
METRICS_DIR  = PROJECT_ROOT / "results" / "metrics"
SUMMARY_PATH = PROJECT_ROOT / "RESULTS_SUMMARY.md"

def load_metrics(path):
    d = json.loads(path.read_text())
    if "test" in d:
        return {
            "accuracy":  d["test"]["accuracy"],
            "precision": d["test"]["precision"],
            "recall":    d["test"]["recall"],
            "f1":        d["test"]["f1"],
            "training_time_min": d["config"]["train_time_s"] / 60,
            "raw": d,
        }
    return {**d, "raw": d}

v1 = load_metrics(METRICS_DIR / "v1_metrics.json")
v2 = load_metrics(METRICS_DIR / "v2_metrics.json")
v3 = load_metrics(METRICS_DIR / "v3_metrics.json")
v4 = load_metrics(METRICS_DIR / "v4_metrics.json")

rows = [
    ("V1 — HOG + SVM",                v1, "Shallow Learning"),
    ("V2 — Custom CNN",               v2, "Deep Learning from scratch"),
    ("V3 — ResNet50 Transfer Learn.", v3, "Supervised TL + fine-tuning"),
    ("V4 — DINOv3 + Linear Probe",    v4, "SSL Foundation Model frozen"),
]

def pct(x): return f"{x*100:.2f}%"

lines = []
lines.append("# Plant Disease Detection — Results Summary")
lines.append("")
lines.append(f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_")
lines.append("")
lines.append("Dataset: **New Plant Diseases Dataset** (augmented PlantVillage, 87,867 images, 38 classes)  ")
lines.append("Split: train 70,295 / val 8,777 / test 8,795 (≈ 80/10/10, valid/ split 50/50 with seed=42)")
lines.append("")

# ── Summary table ────────────────────────────────────
lines.append("## Metrics Table (test set)")
lines.append("")
lines.append("| Version | Approach | Accuracy | Precision | Recall | F1 |")
lines.append("|----------|-----------|:--------:|:---------:|:------:|:--:|")
for name, m, approach in rows:
    lines.append(f"| {name} | {approach} | {pct(m['accuracy'])} | {pct(m['precision'])} | {pct(m['recall'])} | {pct(m['f1'])} |")
lines.append("")

# ── Key Findings ──────────────────────────────────────
lines.append("## Key Findings")
lines.append("")
best = max(rows, key=lambda r: r[1]["accuracy"])
lines.append(f"- **Best overall:** {best[0]} — accuracy {pct(best[1]['accuracy'])}, F1 {pct(best[1]['f1'])}")
gap_v3_v1 = (v3["accuracy"] - v1["accuracy"]) * 100
lines.append(f"- **Gap V1 → V3:** +{gap_v3_v1:.1f} accuracy points. Manual feature extraction (HOG) collapses compared to features learned end-to-end.")
lines.append(f"- **V4 SSL competitive without training:** {pct(v4['accuracy'])} accuracy with a **completely frozen** backbone (0 trainable parameters in the backbone). Just k-NN or a logistic regression on top of the features.")
v2_time = v2["training_time_min"]
v3_time = v3["training_time_min"]
if v3_time < v2_time:
    lines.append(f"- **Transfer learning faster than a custom CNN:** V3 converges in {v3.get('raw', v3).get('best_epoch', '?')} epochs (~{v3_time:.0f} min) vs V2 {v2.get('raw', v2).get('best_epoch','?')} epochs (~{v2_time:.0f} min).")
else:
    lines.append(f"- **Faster per-epoch convergence with TL:** V3 reaches its best at epoch {v3.get('raw', v3).get('best_epoch', '?')}, V2 at epoch {v2.get('raw', v2).get('best_epoch','?')}.")
v4_raw = v4["raw"]
if isinstance(v4_raw, dict) and "knn" in v4_raw and "linear_probe" in v4_raw:
    lines.append(f"- **V4 k-NN vs Linear Probe:** k-NN {pct(v4_raw['knn']['accuracy'])} vs LinProbe {pct(v4_raw['linear_probe']['accuracy'])}. The linear probe exploits the global structure of the feature space better.")
lines.append("")

# ── Augmentation note ─────────────────────────────────
lines.append("> ⚠️ **Note on augmentation:** the dataset is augmented offline, so augmented variants of the same source leaf can end up in both train and val/test. This likely inflates the absolute accuracies (>99%) and should be read as a caveat on the results.")
lines.append("")

# ── Deployment recommendations ──────────────────────────
lines.append("## Deployment Recommendations")
lines.append("")
lines.append("| Scenario | Recommended version | Rationale |")
lines.append("|----------|----------------------|-------------|")
lines.append("| Max accuracy, fixed dataset | **V3 (ResNet50 TL)** | Best absolute score, reasonable parameters (~24M total). |")
lines.append("| Fast onboarding of new classes | **V4 (DINOv3 + linear probe)** | Frozen backbone, just retrain the logistic regression in seconds. |")
lines.append("| Edge / CPU-only / interpretability | **V1 (HOG+SVM)** | Small, inspectable model, but ~75% accuracy. |")
lines.append("| Custom architecture for research / teaching | **V2 (CNN from scratch)** | Full control of the architecture, great for the oral exam. |")
lines.append("")

# ── Reproducibility notes ─────────────────────────────
lines.append("## Reproducibility")
lines.append("")
lines.append("- Fixed seed (`random_state=42`) on split, sklearn, torch.")
lines.append("- Ordered notebooks: `00_setup_and_data` → `06_prepare_documentation`.")
lines.append("- Dependencies in `requirements.txt`. V4 requires `transformers` + HuggingFace login for DINOv3.")
lines.append("- Checkpoints in `results/models/<version>/` (excluded from git via `.gitignore`).")
lines.append("- V4 embedding cache in `results/models/v4_dinov3_probe/embeddings/*.npz` (regenerable in ~13 min).")
lines.append("")

SUMMARY_PATH.write_text("\n".join(lines))
print(f"Saved: {SUMMARY_PATH} ✅")
print(f"\n── Preview first 30 lines ──\n")
print("\n".join(lines[:30]))

Saved: /Users/marco/Documents/repos/ComputerVisionProject/RESULTS_SUMMARY.md ✅

── Preview first 30 lines ──

# Plant Disease Detection — Results Summary

_Generated: 2026-05-31 11:44_

Dataset: **New Plant Diseases Dataset** (augmented PlantVillage, 87,867 images, 38 classes)  
Split: train 70,295 / val 8,777 / test 8,795 (≈ 80/10/10, valid/ split 50/50 with seed=42)

## Metrics Table (test set)

| Version | Approach | Accuracy | Precision | Recall | F1 |
|----------|-----------|:--------:|:---------:|:------:|:--:|
| V1 — HOG + SVM | Shallow Learning | 74.39% | 74.62% | 74.39% | 74.26% |
| V2 — Custom CNN | Deep Learning from scratch | 99.66% | 99.66% | 99.66% | 99.66% |
| V3 — ResNet50 Transfer Learn. | Supervised TL + fine-tuning | 99.87% | 99.88% | 99.87% | 99.87% |
| V4 — DINOv3 + Linear Probe | SSL Foundation Model frozen | 98.35% | 98.39% | 98.35% | 98.35% |

## Key Findings

- **Best overall:** V3 — ResNet50 Transfer Learn. — accuracy 99.87%, F1 99.87%
- **Gap V1 → V3:** +25.5

In [6]:
# ── Cell 3: Export export_for_report.json (unified metrics) ──────
from datetime import datetime

export = {
    "meta": {
        "project":   "Plant Disease Detection",
        "generated": datetime.now().isoformat(timespec="seconds"),
        "dataset":   "New Plant Diseases Dataset (augmented PlantVillage, 87,867 images, 38 classes)",
        "split":     "train 70,295 / val 8,777 / test 8,795 (≈ 80/10/10, valid/ split 50/50, seed=42)",
    },
    "versions": {},
}

for ver, (name, m, approach) in zip(["v1", "v2", "v3", "v4"], rows):
    raw = m["raw"]
    entry = {
        "display_name": name,
        "approach":     approach,
        "metrics": {
            "accuracy":  round(m["accuracy"], 4),
            "precision": round(m["precision"], 4),
            "recall":    round(m["recall"], 4),
            "f1":        round(m["f1"], 4),
        },
    }
    if ver == "v1":
        entry["training_time_min"] = round(raw["config"]["train_time_s"] / 60, 2)
        entry["inference_ms_per_image"] = round(raw["test"]["inference_ms_per_image"], 2)
        entry["hyperparams"] = {"kernel": raw["config"]["svm_kernel"], "C": raw["config"]["svm_C"]}
    elif ver == "v2":
        entry["training_time_min"] = raw.get("training_time_min")
        entry["best_epoch"]        = raw.get("best_epoch")
    elif ver == "v3":
        entry["training_time_min"] = raw.get("training_time_min")
        entry["best_epoch"]        = raw.get("best_epoch")
        entry["trainable_params"]  = raw.get("trainable_params")
        entry["total_params"]      = raw.get("total_params")
    elif ver == "v4":
        entry["backbone"]          = raw.get("backbone")
        entry["embedding_dim"]     = raw.get("embedding_dim")
        entry["extract_time_min"]  = raw.get("extract_time_min")
        entry["backbone_params"]   = raw.get("backbone_params")
        entry["knn"]               = raw.get("knn")
        entry["linear_probe"]      = raw.get("linear_probe")
        entry["best_classifier"]   = raw.get("best_classifier")
    export["versions"][ver] = entry

# Ranking
ranked = sorted(export["versions"].items(),
                key=lambda kv: kv[1]["metrics"]["accuracy"], reverse=True)
export["ranking_by_accuracy"] = [
    {"version": k, "name": v["display_name"], "accuracy": v["metrics"]["accuracy"]}
    for k, v in ranked
]

out_path = METRICS_DIR / "export_for_report.json"
out_path.write_text(json.dumps(export, indent=2))
print(f"Saved: {out_path} ✅")
print("\n── Final ranking by accuracy ──")
for i, e in enumerate(export["ranking_by_accuracy"], 1):
    print(f"  {i}. {e['name']:40s}  accuracy = {e['accuracy']*100:.2f}%")

print("\nDone! Notebook 06 complete — everything ready for the Technical Analysis Document.")

Saved: /Users/marco/Documents/repos/ComputerVisionProject/results/metrics/export_for_report.json ✅

── Final ranking by accuracy ──
  1. V3 — ResNet50 Transfer Learn.             accuracy = 99.87%
  2. V2 — Custom CNN                           accuracy = 99.66%
  3. V4 — DINOv3 + Linear Probe                accuracy = 98.35%
  4. V1 — HOG + SVM                            accuracy = 74.39%

Done! Notebook 06 complete — everything ready for the Technical Analysis Document.
